# Temporary Split Batch 3B

This temporary notebook handles the final smaller portion of batch 3 and writes only its split-specific partial outputs until all three coordinated splits are ready.


## Drive And Repo Setup

Mount Drive, clone the repo into the runtime, and copy the required CSV inputs into the checkout.

- Reads: Google Drive, GitHub repo, runtime filesystem
- Writes: `/content/svg-kaggle-comptetition`, Drive output root
- Rerun-safe: Yes. It reclones the repo and recopies the inputs cleanly.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

GIT_REPO_URL = "https://github.com/Demetri65/svg-kaggle-competition-.git"
CHECKOUT_PATH = Path("/content/svg-kaggle-comptetition")
DRIVE_ROOT = Path("/content/drive/MyDrive")
OUTPUT_ROOT = DRIVE_ROOT / "svg-kaggle-comptetition" / "submission_outputs"

if CHECKOUT_PATH.exists():
    shutil.rmtree(CHECKOUT_PATH)
subprocess.check_call(["git", "clone", GIT_REPO_URL, str(CHECKOUT_PATH)])

for required_name in ["test.csv", "train.csv", "sample_submission.csv"]:
    destination = CHECKOUT_PATH / required_name
    if destination.exists():
        continue

    preferred_sources = [
        DRIVE_ROOT / "svg-kaggle-comptetition" / required_name,
        DRIVE_ROOT / "Colab Notebooks" / "svg-kaggle-comptetition" / required_name,
    ]
    copied = False
    for candidate in preferred_sources:
        if candidate.exists():
            shutil.copy2(candidate, destination)
            copied = True
            break
    if not copied:
        for candidate in DRIVE_ROOT.rglob(required_name):
            if candidate.is_file():
                shutil.copy2(candidate, destination)
                copied = True
                break
    if not copied:
        raise FileNotFoundError(
            f"Could not find {required_name} in Google Drive. Copy it into {CHECKOUT_PATH} manually and rerun this cell."
        )

os.environ["SVG_KAGGLE_REPO_ROOT"] = str(CHECKOUT_PATH)
sys.path.insert(0, str(CHECKOUT_PATH))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repo checkout: {CHECKOUT_PATH}")
print(f"Drive output root: {OUTPUT_ROOT}")


## Package Check

Install any missing runtime packages required by the shared helper module and this notebook.

- Reads: Current Python environment
- Writes: Installed runtime packages
- Rerun-safe: Yes. It only installs missing packages.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "torch": "torch",
    "transformers": "transformers",
    "peft": "peft",
    "accelerate": "accelerate",
    "cairosvg": "cairosvg",
    "skimage": "scikit-image",
    "PIL": "pillow",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print(f"Installed missing packages: {missing_packages}")
else:
    print("All required packages are already installed.")


## Load Helpers, Recommendation, And Split Slice

Import the shared helper module, load the analysis recommendation, prepare the coordinated split-run state, and select this notebook's exact row slice.

- Reads: Repo checkout, helper module, analysis recommendation, competition CSV files
- Writes: Split-run marker and cleared batch 3 artifacts only on first run of this split label
- Rerun-safe: Yes. It reuses the existing split-run marker for the same run label.


In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

import submission_pipeline as pipeline

pipeline.set_global_seed(1337)
PATHS = pipeline.resolve_pipeline_paths(Path(os.environ["SVG_KAGGLE_REPO_ROOT"]), OUTPUT_ROOT)
CONFIG = pipeline.GenerationConfig(verbose_progress=True)
TEST_DF, SAMPLE_SUBMISSION_DF, TRAIN_DF = pipeline.load_competition_frames(PATHS)

print(f"Repo root: {PATHS.repo_root}")
print(f"Output root: {PATHS.output_root}")

BATCH_ID = 3
PART_NAME = "batch_3b"
ROW_START = 700
ROW_END = 750
EXPECTED_PARTS = ('batch_3_main', 'batch_3a', 'batch_3b')
CHECKPOINT_EVERY = 10
SPLIT_RUN_LABEL = "split_batch_3_temp_20260325_v2"

ANALYSIS_RECOMMENDATION = pipeline.load_analysis_recommendation(PATHS)
GENERATION_MODE = ANALYSIS_RECOMMENDATION.get("recommended_generation_mode", "body_only")
BATCH_DIR = pipeline.batch_output_dir(PATHS, BATCH_ID)
PARTIAL_DIR = pipeline.partial_batch_output_dir(PATHS, BATCH_ID)
SLICE_DF = pipeline.select_row_slice(TEST_DF, ROW_START, ROW_END)
INIT_STATUS = pipeline.prepare_split_batch_outputs(
    BATCH_DIR,
    EXPECTED_PARTS,
    run_label=SPLIT_RUN_LABEL,
)

print(f"Recommended generation mode: {GENERATION_MODE}")
print(f"Split run label: {SPLIT_RUN_LABEL}")
print(f"Slice rows: {ROW_START}:{ROW_END} ({len(SLICE_DF)} rows)")
if INIT_STATUS["did_reset"]:
    print("Initialized split batch 3 outputs and cleared stale batch_3 artifacts.")
else:
    print("Using existing split batch 3 partial state for this run label.")
display(SLICE_DF[["id", "prompt", "row_index"]].head())


## Resume From Partial Output

Load this notebook's saved partial results, skip rows that already completed in this split, and show the remaining work.

- Reads: `batch_3/partials/*_results.csv`
- Writes: Nothing
- Rerun-safe: Yes. It is how this notebook resumes after interruption.


In [ ]:
PARTIAL_FRAMES, _ = pipeline.load_partial_batch_outputs(BATCH_DIR, [PART_NAME])
PARTIAL_RESULTS_DF = PARTIAL_FRAMES.get(
    PART_NAME,
    pipeline.normalize_results_df(pd.DataFrame(columns=pipeline.RESULT_COLUMNS)),
)
COMPLETED_IDS = set(PARTIAL_RESULTS_DF["id"].tolist())
PENDING_DF = SLICE_DF[~SLICE_DF["id"].isin(COMPLETED_IDS)].copy().reset_index(drop=True)

print(f"Existing completed rows for {PART_NAME}: {len(PARTIAL_RESULTS_DF)}")
print(f"Pending rows for {PART_NAME}: {len(PENDING_DF)}")
if not PENDING_DF.empty:
    display(PENDING_DF[["id", "row_index"]].head(10))


## Load Model

Load the base model plus the LoRA adapter into the current Colab GPU runtime for first-pass generation.

- Reads: Base model on Hugging Face, LoRA adapter from repo
- Writes: In-memory model state only
- Rerun-safe: No. It is safe to rerun, but it reloads the model and costs time.


In [ ]:
RUNTIME = pipeline.load_lora_runtime(PATHS.adapter_dir)
print(f"Loaded runtime on: {RUNTIME.runtime_device}")


## Deterministic First Pass With Partial Checkpoints

Run one deterministic generation attempt per pending row in this split and checkpoint the split-specific partial CSVs every 10 rows.

- Reads: Selected split rows, model runtime, existing partial outputs
- Writes: `batch_3/partials/*` files for this split only
- Rerun-safe: Yes. It resumes from this split's saved partial results.


In [ ]:
records = PARTIAL_RESULTS_DF.to_dict(orient="records")
total_pending = len(PENDING_DF)

if total_pending == 0:
    print(f"{PART_NAME} is already complete. Rewriting the current partial artifacts for consistency.")
else:
    for local_index, row in enumerate(PENDING_DF.to_dict(orient="records"), start=1):
        row_id = row["id"]
        prompt = row["prompt"]
        row_index = int(row["row_index"])
        started_at = pipeline.time.perf_counter()
        try:
            prepared = pipeline.infer_svg_candidate(
                prompt,
                RUNTIME,
                generation_mode=GENERATION_MODE,
                config=CONFIG,
                allow_sampled_repair=False,
                progress_label=f"{PART_NAME} {local_index}/{total_pending}",
                row_id=row_id,
            )
            runtime_seconds = pipeline.time.perf_counter() - started_at
            record = pipeline.build_success_record(
                row_id,
                prompt,
                prepared,
                batch_id=str(BATCH_ID),
                row_index=row_index,
                generation_mode=GENERATION_MODE,
                first_pass_success=True,
                runtime_seconds=runtime_seconds,
            )
        except pipeline.RowInferenceError as exc:
            runtime_seconds = pipeline.time.perf_counter() - started_at
            record = pipeline.build_failed_record(
                row_id,
                prompt,
                exc,
                batch_id=str(BATCH_ID),
                row_index=row_index,
                generation_mode=GENERATION_MODE,
                first_pass_success=False,
                runtime_seconds=runtime_seconds,
            )
        records.append(record)
        if local_index % CHECKPOINT_EVERY == 0 or local_index == total_pending:
            PARTIAL_RESULTS_DF = pipeline.normalize_results_df(pd.DataFrame(records))
            PARTIAL_RESULTS_DF = PARTIAL_RESULTS_DF.sort_values("row_index").drop_duplicates(subset=["id"], keep="last")
            pipeline.save_partial_batch_outputs(PARTIAL_RESULTS_DF, BATCH_DIR, PART_NAME)
            print(
                f"Checkpointed {PART_NAME}: {len(PARTIAL_RESULTS_DF)}/{len(SLICE_DF)} rows saved to {PARTIAL_DIR}"
            )
        pipeline.release_runtime_memory()

PARTIAL_RESULTS_DF = pipeline.normalize_results_df(pd.DataFrame(records))
PARTIAL_RESULTS_DF = PARTIAL_RESULTS_DF.sort_values("row_index").drop_duplicates(subset=["id"], keep="last")
pipeline.save_partial_batch_outputs(PARTIAL_RESULTS_DF, BATCH_DIR, PART_NAME)
print(f"Wrote partial batch artifacts to: {PARTIAL_DIR}")


## Split Summary

Show the current success and failure totals for this split of batch 3.

- Reads: In-memory partial results
- Writes: Nothing
- Rerun-safe: Yes. Read-only display cell.


In [ ]:
display(pipeline.summarize_batch_results(PARTIAL_RESULTS_DF))
FAILED_DF = PARTIAL_RESULTS_DF[~PARTIAL_RESULTS_DF["first_pass_success"]].copy()
if FAILED_DF.empty:
    print("No failed rows in this split.")
else:
    display(FAILED_DF[["id", "row_index", "generation_mode", "gate_error"]].head(20))


## Finalize Canonical Batch 3 Outputs

If all three split notebooks have finished, combine the partial outputs into the normal `batch_3/*.csv` files expected by the merge notebook. Otherwise print which split is still missing.

- Reads: `batch_3/partials/*`
- Writes: Canonical `batch_3/*.csv` only when all three splits are present
- Rerun-safe: Yes. It rewrites the canonical batch 3 files if all splits are ready.


In [ ]:
FINALIZE_STATUS = pipeline.finalize_split_batch_outputs(BATCH_DIR, expected_row_count=250)
if FINALIZE_STATUS["ready"]:
    print("Canonical batch_3 files are now finalized.")
    print(
        f"Rows: {FINALIZE_STATUS['rows']} | row range: {FINALIZE_STATUS['first_row_index']}:{FINALIZE_STATUS['last_row_index']}"
    )
else:
    print("Waiting for the other split notebooks before writing canonical batch_3 files.")
    print(f"Missing partials: {FINALIZE_STATUS['missing_parts']}")
    print(f"Currently saved rows across available partials: {FINALIZE_STATUS['rows']}")
display(pipeline.artifact_status_table(PATHS, batch_count=4))
